In [ ]:
import os
import zipfile
import random
import shutil
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns


print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
from google.colab import files
files.upload()  # kaggle.json

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install kaggle

In [ ]:
# Chest X-Ray Pneumonia dataset
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
import zipfile
import os

with zipfile.ZipFile('chest-xray-pneumonia.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/chest_xray')

# Verify directory structure
os.listdir('/content/chest_xray/chest_xray')

In [ ]:
import os
import random
import shutil
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# paths
base_dir = '/content/chest_xray/chest_xray'  # Corrected path
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Verify directories
print("Train dir exists:", os.path.exists(train_dir))
print("Test dir exists:", os.path.exists(test_dir))
if os.path.exists(train_dir):
    print("Train subdirs:", os.listdir(train_dir))

# Subsample dataset (300 train + 60 val per class for CPU)
sub_train_dir = '/content/sub_train'
sub_val_dir = '/content/sub_val'
os.makedirs(sub_train_dir, exist_ok=True)
os.makedirs(sub_val_dir, exist_ok=True)

def subsample_dir(src_dir, dest_dir, class_name, num_samples):
    src_path = os.path.join(src_dir, class_name)
    dest_path = os.path.join(dest_dir, class_name)
    if not os.path.exists(src_path):
        print(f"Error: {src_path} not found. Skipping {class_name}.")
        return False
    os.makedirs(dest_path, exist_ok=True)
    files = os.listdir(src_path)
    if not files:
        print(f"Error: No files in {src_path}.")
        return False
    random.shuffle(files)
    for file in files[:num_samples]:
        shutil.copy(os.path.join(src_path, file), os.path.join(dest_path, file))
    print(f"Copied {min(num_samples, len(files))} files for {class_name} to {dest_path}")
    return True

classes = ['NORMAL', 'PNEUMONIA']
success = True
for cls in classes:
    if not subsample_dir(train_dir, sub_train_dir, cls, 300):
        success = False
    if not subsample_dir(train_dir, sub_val_dir, cls, 60):
        success = False

if not success:
    raise FileNotFoundError("Failed to subsample dataset. Check directory structure.")

# Data augmentation
img_size = (128, 128)
batch_size = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    sub_train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    sub_val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = val_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

print(f"Train samples: {train_generator.samples}, Val samples: {val_generator.samples}, Test samples: {test_generator.samples}")
if train_generator.samples == 0 or val_generator.samples == 0:
    raise ValueError("No images found in train/val directories. Check subsampling.")


def plot_class_distribution(generator, title):
    labels = generator.classes
    unique, counts = np.unique(labels, return_counts=True)
    df = pd.DataFrame({'Class': ['NORMAL', 'PNEUMONIA'], 'Count': counts})
    plt.figure(figsize=(6, 4))
    sns.barplot(x='Class', y='Count', hue='Class', data=df, palette=['#1f77b4', '#ff7f0e'], legend=False)
    plt.title(title)
    plt.show()

plot_class_distribution(train_generator, 'Train Class Distribution')
plot_class_distribution(val_generator, 'Val Class Distribution')

# Build model
base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = Sequential([
    base_model,
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.0005),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)
]

# Train (3 epochs for CPU; ~15

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from flask import Flask, request, jsonify
import io
from PIL import Image

def plot_class_distribution(generator, title):
    labels = generator.classes
    unique, counts = np.unique(labels, return_counts=True)
    df = pd.DataFrame({'Class': ['NORMAL', 'PNEUMONIA'], 'Count': counts})
    plt.figure(figsize=(6, 4))
    sns.barplot(x='Class', y='Count', hue='Class', data=df, palette=['#1f77b4', '#ff7f0e'], legend=False)
    plt.title(title)
    plt.show()


plot_class_distribution(train_generator, 'Train Class Distribution')
plot_class_distribution(val_generator, 'Val Class Distribution')

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)
]

print("Starting training...")
history = model.fit(
    train_generator,
    epochs=3,
    validation_data=val_generator,
    callbacks=callbacks
)
print("Training complete.")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc', color='#1f77b4')
plt.plot(history.history['val_accuracy'], label='Val Acc', color='#ff7f0e')
plt.title('Model Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='#1f77b4')
plt.plot(history.history['val_loss'], label='Val Loss', color='#ff7f0e')
plt.title('Model Loss')
plt.legend()
plt.show()


model.load_weights('best_model.h5')
test_loss, test_acc, test_precision, test_recall = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc:.2f}, Precision: {test_precision:.2f}, Recall: {test_recall:.2f}")

# Confusion matrix
y_pred = (model.predict(test_generator) > 0.5).astype(int)
y_true = test_generator.classes
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['NORMAL', 'PNEUMONIA'], yticklabels=['NORMAL', 'PNEUMONIA'])
plt.title('Confusion Matrix')
plt.show()
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))


model.save('pneumonia_model.h5')

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    file = request.files['image']
    img = Image.open(io.BytesIO(file.read())).resize((128, 128))
    img = np.array(img) / 255.0
    if img.shape[-1] == 4:
        img = img[:, :, :3]
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img)[0][0]
    return jsonify({'prediction': 'PNEUMONIA' if pred > 0.5 else 'NORMAL', 'confidence': float(pred)})



## Enhanced version

In [ ]:
!pip install keras-tuner --quiet


In [ ]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import os
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix


base_dir = '/content/chest_xray/chest_xray'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')


print("Train dir exists:", os.path.exists(train_dir))
print("Test dir exists:", os.path.exists(test_dir))
if os.path.exists(train_dir):
    print("Train subdirs:", os.listdir(train_dir))

# Step 5: Subsample Dataset (250 train + 50 val per class)
sub_train_dir = '/content/sub_train'
sub_val_dir = '/content/sub_val'
os.makedirs(sub_train_dir, exist_ok=True)
os.makedirs(sub_val_dir, exist_ok=True)

def subsample_dir(src_dir, dest_dir, class_name, num_samples):
    src_path = os.path.join(src_dir, class_name)
    dest_path = os.path.join(dest_dir, class_name)
    if not os.path.exists(src_path):
        print(f"Error: {src_path} not found. Skipping {class_name}.")
        return False
    os.makedirs(dest_path, exist_ok=True)
    files = os.listdir(src_path)
    if not files:
        print(f"Error: No files in {src_path}.")
        return False
    random.shuffle(files)
    for file in files[:num_samples]:
        shutil.copy(os.path.join(src_path, file), os.path.join(dest_path, file))
    print(f"Copied {min(num_samples, len(files))} files for {class_name} to {dest_path}")
    return True

classes = ['NORMAL', 'PNEUMONIA']
success = True
for cls in classes:
    if not subsample_dir(train_dir, sub_train_dir, cls, 250):
        success = False
    if not subsample_dir(train_dir, sub_val_dir, cls, 50):
        success = False

if not success:
    raise FileNotFoundError("Failed to subsample dataset. Check directory structure.")


img_size = (128, 128)
batch_size = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    sub_train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    sub_val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary'
)

test_generator = val_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)

print(f"Train samples: {train_generator.samples}, Val samples: {val_generator.samples}, Test samples: {test_generator.samples}")
if train_generator.samples == 0 or val_generator.samples == 0:
    raise ValueError("No images found in train/val directories. Check subsampling.")


def plot_class_distribution(generator, title):
    labels = generator.classes
    unique, counts = np.unique(labels, return_counts=True)
    df = pd.DataFrame({'Class': ['NORMAL', 'PNEUMONIA'], 'Count': counts})
    plt.figure(figsize=(6, 4))
    sns.barplot(x='Class', y='Count', hue='Class', data=df, palette=['#1f77b4', '#ff7f0e'], legend=False)
    plt.title(title)
    plt.show()

plot_class_distribution(train_generator, 'Train Class Distribution')
plot_class_distribution(val_generator, 'Val Class Distribution')


def build_model(hp):
    base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    model = Sequential([
        base_model,
        Flatten(),
        Dense(hp.Int('units', 32, 128, step=32), activation='relu'),
        Dropout(hp.Float('dropout', 0.3, 0.7, step=0.1)),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(hp.Float('lr', 1e-4, 1e-2, sampling='log')),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    return model


tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=3,
    directory='tuner_dir',
    project_name='pneumonia'
)
callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)
]
tuner.search(train_generator, epochs=3, validation_data=val_generator, callbacks=callbacks)


best_model = tuner.get_best_models(num_models=1)[0]


class_weight = {0: 1.5, 1: 1.0}

print("Starting fine-tuning with class weights...")
history = best_model.fit(
    train_generator,
    epochs=3,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weight
)
print("Training complete.")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc', color='#1f77b4')
plt.plot(history.history['val_accuracy'], label='Val Acc', color='#ff7f0e')
plt.title('Model Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='#1f77b4')
plt.plot(history.history['val_loss'], label='Val Loss', color='#ff7f0e')
plt.title('Model Loss')
plt.legend()
plt.show()


test_loss, test_acc, test_precision, test_recall = best_model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc:.2f}, Precision: {test_precision:.2f}, Recall: {test_recall:.2f}")


y_pred = (best_model.predict(test_generator) > 0.5).astype(int)
y_true = test_generator.classes
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['NORMAL', 'PNEUMONIA'], yticklabels=['NORMAL', 'PNEUMONIA'])
plt.title('Confusion Matrix')
plt.show()
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

best_model.save('enhanced_pneumonia_model.keras')

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from flask import Flask, request, jsonify
import io
from PIL import Image

tf.keras.backend.clear_session()
print("Keras session cleared.")
print("Building model from scratch...")
input_shape = (128, 128, 3)
base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
base_model.trainable = False
inputs = tf.keras.Input(shape=input_shape)
x = base_model(inputs, training=False)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(32, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("Model built with input shape:", model.input_shape)


print("\nModel Summary:")
model.summary()
print("\nMobileNetV2 (base model) Summary:")
base_model.summary()


def get_gradcam_heatmap(model, img_array, last_conv_layer_name='block_16_project'):
    base_model = model.layers[1]
    try:
        conv_layer = base_model.get_layer(last_conv_layer_name)
        print(f"Using layer: {last_conv_layer_name} (Type: {conv_layer.__class__.__name__})")
    except ValueError:
        print(f"Layer {last_conv_layer_name} not found in MobileNetV2. Searching for last conv layer...")
        last_conv_layer_name = None
        for layer in base_model.layers[::-1]:
            if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
                last_conv_layer_name = layer.name
                print(f"Found conv layer: {last_conv_layer_name}")
                break
        if last_conv_layer_name is None:
            raise ValueError(f"No convolutional layer found in MobileNetV2. Layers: {[layer.name for layer in base_model.layers]}")
        conv_layer = base_model.get_layer(last_conv_layer_name)

    print("Warming up model with a prediction...")
    dummy_input = np.zeros((1, 128, 128, 3), dtype=np.float32)
    _ = model.predict(dummy_input, verbose=0)
    print("Model warm-up complete.")

    print(f"Model input shape: {model.input.shape}")
    print(f"Base model input shape: {base_model.input.shape}")
    try:
        print(f"Conv layer output shape: {conv_layer.output.shape}")
    except AttributeError:
        print("Conv layer output shape not available yet (will be computed during Grad-CAM).")
    print(f"Model output shape: {model.output.shape}")

    try:
        inputs = base_model.input  # Use MobileNetV2 input
        conv_output = base_model.get_layer(last_conv_layer_name).output
        x = base_model.output
        for layer in model.layers[2:]:  # Apply Flatten, Dense, Dropout, Dense
            x = layer(x)
        final_output = x
        grad_model = tf.keras.models.Model(inputs=[inputs], outputs=[conv_output, final_output])
    except Exception as e:
        print(f"Error creating Grad-CAM model: {e}")
        raise

    print("\nGrad-CAM Model Summary:")
    grad_model.summary()

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        raise ValueError(f"Gradients are None for layer {last_conv_layer_name}. Check if the layer is part of the computation graph.")
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(conv_outputs, pooled_grads), axis=-1)
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (np.max(heatmap) + 1e-10)
    return heatmap

# Load test generator
val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = val_datagen.flow_from_directory(
    '/content/chest_xray/chest_xray/test',
    target_size=(128, 128),
    batch_size=16,
    class_mode='binary',
    shuffle=False
)
print(f"Test generator loaded: {test_generator.samples} samples")

test_img, test_label = next(test_generator)
print(f"Test image batch shape: {test_img.shape}")

for i in range(3):
    img_array = test_img[i:i+1]
    print(f"Generating Grad-CAM for image {i+1}, shape: {img_array.shape}")
    try:
        heatmap = get_gradcam_heatmap(model, img_array, last_conv_layer_name='block_16_project')
    except ValueError as e:
        print(f"Grad-CAM error: {e}")
        raise
    img = test_img[i] * 255
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    superimposed_img = heatmap * 0.4 + img

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img.astype('uint8'))
    plt.title('Original Image' + (' (PNEUMONIA)' if test_label[i] > 0.5 else ' (NORMAL)'))
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(superimposed_img.astype('uint8'))
    plt.title('Grad-CAM Heatmap')
    plt.axis('off')
    plt.show()

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    file = request.files['image']
    img = Image.open(io.BytesIO(file.read())).resize((128, 128))
    img = np.array(img) / 255.0
    if img.shape[-1] == 4:
        img = img[:, :, :3]
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img)[0][0]
    return jsonify({'prediction': 'PNEUMONIA' if pred > 0.5 else 'NORMAL', 'confidence': float(pred)})


In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model('enhanced_pneumonia_model.keras')

base_model = model.layers[0]

print("MobileNetV2 Layer Names:")
for i, layer in enumerate(base_model.layers):
    print(f"Layer {i}: {layer.name} (Type: {layer.__class__.__name__})")

conv_layers = [layer.name for layer in base_model.layers if 'conv' in layer.name.lower() or 'block' in layer.name.lower()]
print("\nPotential Convolutional Layers for Grad-CAM:")
for i, name in enumerate(conv_layers):
    print(f"{i}: {name}")

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator


tf.keras.backend.clear_session()
print("Keras session cleared.")

try:
    model = tf.keras.models.load_model('enhanced_pneumonia_model.keras')
    print("Model loaded successfully.")
    dummy_input = np.zeros((1, 128, 128, 3), dtype=np.float32)
    model(dummy_input)
except Exception as e:
    print(f"Error loading model: {e}")
    print("Rebuilding model from scratch as fallback...")
    base_model = tf.keras.applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    inputs = tf.keras.Input(shape=(128, 128, 3))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Create Grad-CAM model
def create_gradcam_model(model, last_conv_layer_name='block_16_project'):
    base_model = model.layers[0] if isinstance(model, tf.keras.Sequential) else model.layers[1]
    try:
        conv_layer = base_model.get_layer(last_conv_layer_name)
        print(f"Using layer: {last_conv_layer_name}")
    except ValueError:
        print(f"Layer {last_conv_layer_name} not found. Searching for last conv layer...")
        last_conv_layer_name = None
        for layer in base_model.layers[::-1]:
            if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
                last_conv_layer_name = layer.name
                print(f"Found conv layer: {last_conv_layer_name}")
                break
        if last_conv_layer_name is None:
            raise ValueError("No convolutional layer found in MobileNetV2.")
        conv_layer = base_model.get_layer(last_conv_layer_name)

    inputs = base_model.input
    conv_output = base_model.get_layer(last_conv_layer_name).output
    x = base_model.output
    top_layers = model.layers[1:] if isinstance(model, tf.keras.Sequential) else model.layers[2:]
    for layer in top_layers:
        x = layer(x)
    final_output = x
    grad_model = tf.keras.models.Model(inputs=[inputs], outputs=[conv_output, final_output])
    return grad_model

# Grad-CAM heatmap function
def get_gradcam_heatmap(grad_model, img_array):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        print("Warning: Gradients are None.")
        return None, predictions[0, 0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(conv_outputs, pooled_grads), axis=-1)
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) < 1e-10:
        print("Warning: Heatmap max is near zero, skipping normalization.")
        return None, predictions[0, 0]
    heatmap = heatmap / (np.max(heatmap) + 1e-10)
    return heatmap, predictions[0, 0]

# Load test generator
val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = val_datagen.flow_from_directory(
    '/content/chest_xray/chest_xray/test',
    target_size=(128, 128),
    batch_size=1,
    class_mode='binary',
    shuffle=False
)
print(f"Test generator loaded: {test_generator.samples} samples")
print(f"Class indices: {test_generator.class_indices}")

# Create Grad-CAM model
grad_model = create_gradcam_model(model)

# Collect one NORMAL and one PNEUMONIA image
normal_img, pneumonia_img = None, None
normal_label, pneumonia_label = None, None
for i in range(test_generator.samples):
    img, label = next(test_generator)
    if label[0] == 0 and normal_img is None:  # NORMAL
        normal_img = img
        normal_label = label[0]
    elif label[0] == 1 and pneumonia_img is None:  # PNEUMONIA
        pneumonia_img = img
        pneumonia_label = label[0]
    if normal_img is not None and pneumonia_img is not None:
        break

# Generate heatmaps for both images
if normal_img is None or pneumonia_img is None:
    raise ValueError("Could not find both NORMAL and PNEUMONIA images in test set.")

# Generate Grad-CAM for NORMAL image
normal_heatmap, normal_pred = get_gradcam_heatmap(grad_model, normal_img)
if normal_heatmap is None:
    raise ValueError("Failed to generate heatmap for NORMAL image.")

# Generate Grad-CAM for PNEUMONIA image
pneumonia_heatmap, pneumonia_pred = get_gradcam_heatmap(grad_model, pneumonia_img)
if pneumonia_heatmap is None:
    raise ValueError("Failed to generate heatmap for PNEUMONIA image.")

# Prepare images and heatmaps for visualization
normal_img_display = normal_img[0] * 255
pneumonia_img_display = pneumonia_img[0] * 255

# Resize heatmaps and apply colormap
normal_heatmap = cv2.resize(normal_heatmap, (128, 128))
normal_heatmap = np.uint8(255 * normal_heatmap)
normal_heatmap = cv2.applyColorMap(normal_heatmap, cv2.COLORMAP_JET)
normal_superimposed = normal_heatmap * 0.6 + normal_img_display * 0.4

pneumonia_heatmap = cv2.resize(pneumonia_heatmap, (128, 128))
pneumonia_heatmap = np.uint8(255 * pneumonia_heatmap)
pneumonia_heatmap = cv2.applyColorMap(pneumonia_heatmap, cv2.COLORMAP_JET)
pneumonia_superimposed = pneumonia_heatmap * 0.6 + pneumonia_img_display * 0.4

# Plot side by side
plt.figure(figsize=(15, 5))

# NORMAL image
plt.subplot(1, 2, 1)
plt.imshow(normal_img_display.astype('uint8'))
plt.title(f"Normal Image\nPrediction: {'NORMAL' if normal_pred < 0.5 else 'PNEUMONIA'} (Conf: {normal_pred:.4f})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(normal_superimposed.astype('uint8'))
plt.title('Normal Image with Grad-CAM Heatmap')
plt.axis('off')

plt.figure(figsize=(15, 5))

# PNEUMONIA image
plt.subplot(1, 2, 1)
plt.imshow(pneumonia_img_display.astype('uint8'))
plt.title(f"Pneumonia Image\nPrediction: {'PNEUMONIA' if pneumonia_pred > 0.5 else 'NORMAL'} (Conf: {pneumonia_pred:.4f})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(pneumonia_superimposed.astype('uint8'))
plt.title('Pneumonia Image with Grad-CAM Heatmap')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Clear Keras session
tf.keras.backend.clear_session()
print("Keras session cleared.")

# Load the trained model
try:
    model = tf.keras.models.load_model('enhanced_pneumonia_model.keras')
    print("Model loaded successfully.")
    dummy_input = np.zeros((1, 128, 128, 3), dtype=np.float32)
    model(dummy_input)
except Exception as e:
    print(f"Error loading model: {e}")
    print("Rebuilding model as fallback...")
    base_model = tf.keras.applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    inputs = tf.keras.Input(shape=(128, 128, 3))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Create Grad-CAM model
def create_gradcam_model(model, last_conv_layer_name='block_13_project'):
    base_model = model.layers[0] if isinstance(model, tf.keras.Sequential) else model.layers[1]
    try:
        conv_layer = base_model.get_layer(last_conv_layer_name)
        print(f"Using layer: {last_conv_layer_name}")
    except ValueError:
        print(f"Layer {last_conv_layer_name} not found. Searching for last conv layer...")
        last_conv_layer_name = None
        for layer in base_model.layers[::-1]:
            if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
                last_conv_layer_name = layer.name
                print(f"Found conv layer: {last_conv_layer_name}")
                break
        if last_conv_layer_name is None:
            raise ValueError("No convolutional layer found in MobileNetV2.")
        conv_layer = base_model.get_layer(last_conv_layer_name)

    inputs = base_model.input
    conv_output = base_model.get_layer(last_conv_layer_name).output
    x = base_model.output
    top_layers = model.layers[1:] if isinstance(model, tf.keras.Sequential) else model.layers[2:]
    for layer in top_layers:
        x = layer(x)
    final_output = x
    grad_model = tf.keras.models.Model(inputs=[inputs], outputs=[conv_output, final_output])
    return grad_model

# Grad-CAM heatmap function
def get_gradcam_heatmap(grad_model, img_array):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        print("Warning: Gradients are None.")
        return None, predictions[0, 0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(conv_outputs, pooled_grads), axis=-1)
    heatmap = np.maximum(heatmap, 0)
    print(f"Heatmap stats - Min: {np.min(heatmap):.4f}, Max: {np.max(heatmap):.4f}, Mean: {np.mean(heatmap):.4f}")
    if np.max(heatmap) < 1e-10:
        print("Warning: Heatmap max is near zero, skipping normalization.")
        return None, predictions[0, 0]
    heatmap = heatmap / (np.max(heatmap) + 1e-10)
    return heatmap, predictions[0, 0]

# Load test generator
val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = val_datagen.flow_from_directory(
    '/content/chest_xray/chest_xray/test',
    target_size=(128, 128),
    batch_size=1,
    class_mode='binary',
    shuffle=False
)
print(f"Test generator loaded: {test_generator.samples} samples")
print(f"Class indices: {test_generator.class_indices}")

# Create Grad-CAM model
grad_model = create_gradcam_model(model)

# Collect two NORMAL and two PNEUMONIA images
normal_imgs, pneumonia_imgs = [], []
normal_labels, pneumonia_labels = [], []
normal_count, pneumonia_count = 0, 0
for i in range(test_generator.samples):
    img, label = next(test_generator)
    if label[0] == 0 and normal_count < 2:  # NORMAL
        normal_imgs.append(img)
        normal_labels.append(label[0])
        normal_count += 1
    elif label[0] == 1 and pneumonia_count < 2:  # PNEUMONIA
        pneumonia_imgs.append(img)
        pneumonia_labels.append(label[0])
        pneumonia_count += 1
    if normal_count == 2 and pneumonia_count == 2:
        break

if len(normal_imgs) < 2 or len(pneumonia_imgs) < 2:
    raise ValueError("Could not find enough NORMAL or PNEUMONIA images in test set.")

# Generate and plot heatmaps for two images per class
for idx in range(2):
    # Generate heatmaps
    normal_heatmap, normal_pred = get_gradcam_heatmap(grad_model, normal_imgs[idx])
    if normal_heatmap is None:
        print(f"Skipping NORMAL image {idx+1} due to invalid heatmap.")
        continue
    pneumonia_heatmap, pneumonia_pred = get_gradcam_heatmap(grad_model, pneumonia_imgs[idx])
    if pneumonia_heatmap is None:
        print(f"Skipping PNEUMONIA image {idx+1} due to invalid heatmap.")
        continue

    # Prepare images and heatmaps
    normal_img_display = normal_imgs[idx][0] * 255
    pneumonia_img_display = pneumonia_imgs[idx][0] * 255

    normal_heatmap = cv2.resize(normal_heatmap, (128, 128))
    normal_heatmap = np.uint8(255 * normal_heatmap)
    normal_heatmap = cv2.applyColorMap(normal_heatmap, cv2.COLORMAP_JET)
    normal_superimposed = normal_heatmap * 0.6 + normal_img_display * 0.4

    pneumonia_heatmap = cv2.resize(pneumonia_heatmap, (128, 128))
    pneumonia_heatmap = np.uint8(255 * pneumonia_heatmap)
    pneumonia_heatmap = cv2.applyColorMap(pneumonia_heatmap, cv2.COLORMAP_JET)
    pneumonia_superimposed = pneumonia_heatmap * 0.6 + pneumonia_img_display * 0.4

    # Plot side by side
    plt.figure(figsize=(15, 5))

    plt.subplot(2, 2, 1)
    plt.imshow(normal_img_display.astype('uint8'))
    plt.title(f"Normal Image {idx+1}\nPred: {'NORMAL' if normal_pred < 0.5 else 'PNEUMONIA'} (Conf: {normal_pred:.4f})")
    plt.axis('off')

    plt.subplot(2, 2, 2)
    plt.imshow(normal_superimposed.astype('uint8'))
    plt.title(f"Normal Grad-CAM Heatmap {idx+1}")
    plt.axis('off')

    plt.subplot(2, 2, 3)
    plt.imshow(pneumonia_img_display.astype('uint8'))
    plt.title(f"Pneumonia Image {idx+1}\nPred: {'PNEUMONIA' if pneumonia_pred > 0.5 else 'NORMAL'} (Conf: {pneumonia_pred:.4f})")
    plt.axis('off')

    plt.subplot(2, 2, 4)
    plt.imshow(pneumonia_superimposed.astype('uint8'))
    plt.title(f"Pneumonia Grad-CAM Heatmap {idx+1}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Clear Keras session
tf.keras.backend.clear_session()
print("Keras session cleared.")

# Load the trained model
try:
    model = tf.keras.models.load_model('enhanced_pneumonia_model.keras')
    print("Model loaded successfully.")
    dummy_input = np.zeros((1, 128, 128, 3), dtype=np.float32)
    model(dummy_input)
except Exception as e:
    print(f"Error loading model: {e}")
    print("Rebuilding model as fallback...")
    base_model = tf.keras.applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    inputs = tf.keras.Input(shape=(128, 128, 3))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Grad-CAM model with a shallower layer
def create_gradcam_model(model, last_conv_layer_name='block_13_project'):
    base_model = model.layers[0] if isinstance(model, tf.keras.Sequential) else model.layers[1]
    try:
        conv_layer = base_model.get_layer(last_conv_layer_name)
        print(f"Using layer: {last_conv_layer_name}")
    except ValueError:
        print(f"Layer {last_conv_layer_name} not found. Searching for last conv layer...")
        last_conv_layer_name = None
        for layer in base_model.layers[::-1]:
            if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
                last_conv_layer_name = layer.name
                print(f"Found conv layer: {last_conv_layer_name}")
                break
        if last_conv_layer_name is None:
            raise ValueError("No convolutional layer found in MobileNetV2.")
        conv_layer = base_model.get_layer(last_conv_layer_name)

    inputs = base_model.input
    conv_output = base_model.get_layer(last_conv_layer_name).output
    x = base_model.output
    top_layers = model.layers[1:] if isinstance(model, tf.keras.Sequential) else model.layers[2:]
    for layer in top_layers:
        x = layer(x)
    final_output = x
    grad_model = tf.keras.models.Model(inputs=[inputs], outputs=[conv_output, final_output])
    return grad_model

# Improved Grad-CAM heatmap function
def get_gradcam_heatmap(grad_model, img_array):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        print("Warning: Gradients are None.")
        return None, predictions[0, 0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(conv_outputs, pooled_grads), axis=-1)
    heatmap = np.maximum(heatmap, 0)
    print(f"Heatmap stats - Min: {np.min(heatmap):.4f}, Max: {np.max(heatmap):.4f}, Mean: {np.mean(heatmap):.4f}")
    if np.max(heatmap) < 1e-10:
        print("Warning: Heatmap max is near zero, skipping normalization.")
        return None, predictions[0, 0]
    heatmap = heatmap / (np.max(heatmap) + 1e-10)
    return heatmap, predictions[0, 0]

# Load test generator
val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = val_datagen.flow_from_directory(
    '/content/chest_xray/chest_xray/test',
    target_size=(128, 128),
    batch_size=1,
    class_mode='binary',
    shuffle=False
)
print(f"Test generator loaded: {test_generator.samples} samples")
print(f"Class indices: {test_generator.class_indices}")

# Create Grad-CAM model
grad_model = create_gradcam_model(model)

# Collect one NORMAL and one PNEUMONIA image
normal_img, pneumonia_img = None, None
normal_label, pneumonia_label = None, None
for i in range(test_generator.samples):
    img, label = next(test_generator)
    if label[0] == 0 and normal_img is None:  # NORMAL
        normal_img = img
        normal_label = label[0]
    elif label[0] == 1 and pneumonia_img is None:  # PNEUMONIA
        pneumonia_img = img
        pneumonia_label = label[0]
    if normal_img is not None and pneumonia_img is not None:
        break

if normal_img is None or pneumonia_img is None:
    raise ValueError("Could not find both NORMAL and PNEUMONIA images in test set.")

# Generate heatmaps
normal_heatmap, normal_pred = get_gradcam_heatmap(grad_model, normal_img)
if normal_heatmap is None:
    raise ValueError("Failed to generate heatmap for NORMAL image.")
pneumonia_heatmap, pneumonia_pred = get_gradcam_heatmap(grad_model, pneumonia_img)
if pneumonia_heatmap is None:
    raise ValueError("Failed to generate heatmap for PNEUMONIA image.")

# Prepare images and heatmaps
normal_img_display = normal_img[0] * 255
pneumonia_img_display = pneumonia_img[0] * 255

normal_heatmap = cv2.resize(normal_heatmap, (128, 128))
normal_heatmap = np.uint8(255 * normal_heatmap)
normal_heatmap = cv2.applyColorMap(normal_heatmap, cv2.COLORMAP_JET)
normal_superimposed = normal_heatmap * 0.6 + normal_img_display * 0.4

pneumonia_heatmap = cv2.resize(pneumonia_heatmap, (128, 128))
pneumonia_heatmap = np.uint8(255 * pneumonia_heatmap)
pneumonia_heatmap = cv2.applyColorMap(pneumonia_heatmap, cv2.COLORMAP_JET)
pneumonia_superimposed = pneumonia_heatmap * 0.6 + pneumonia_img_display * 0.4

# Plot side by side
plt.figure(figsize=(15, 5))

plt.subplot(2, 2, 1)
plt.imshow(normal_img_display.astype('uint8'))
plt.title(f"Normal Image\nPred: {'NORMAL' if normal_pred < 0.5 else 'PNEUMONIA'} (Conf: {normal_pred:.4f})")
plt.axis('off')

plt.subplot(2, 2, 2)
plt.imshow(normal_superimposed.astype('uint8'))
plt.title('Normal Grad-CAM Heatmap')
plt.axis('off')

plt.subplot(2, 2, 3)
plt.imshow(pneumonia_img_display.astype('uint8'))
plt.title(f"Pneumonia Image\nPred: {'PNEUMONIA' if pneumonia_pred > 0.5 else 'NORMAL'} (Conf: {pneumonia_pred:.4f})")
plt.axis('off')

plt.subplot(2, 2, 4)
plt.imshow(pneumonia_superimposed.astype('uint8'))
plt.title('Pneumonia Grad-CAM Heatmap')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Clear Keras session
tf.keras.backend.clear_session()
print("Keras session cleared.")

# Load the trained model
try:
    model = tf.keras.models.load_model('enhanced_pneumonia_model.keras')
    print("Model loaded successfully.")
    dummy_input = np.zeros((1, 128, 128, 3), dtype=np.float32)
    model(dummy_input)
except Exception as e:
    print(f"Error loading model: {e}")
    print("Rebuilding model as fallback...")
    base_model = tf.keras.applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    inputs = tf.keras.Input(shape=(128, 128, 3))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Grad-CAM model with a shallower layer
def create_gradcam_model(model, last_conv_layer_name='block_10_project'):
    base_model = model.layers[0] if isinstance(model, tf.keras.Sequential) else model.layers[1]
    try:
        conv_layer = base_model.get_layer(last_conv_layer_name)
        print(f"Using layer: {last_conv_layer_name}")
    except ValueError:
        print(f"Layer {last_conv_layer_name} not found. Searching for last conv layer...")
        last_conv_layer_name = None
        for layer in base_model.layers[::-1]:
            if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
                last_conv_layer_name = layer.name
                print(f"Found conv layer: {last_conv_layer_name}")
                break
        if last_conv_layer_name is None:
            raise ValueError("No convolutional layer found in MobileNetV2.")
        conv_layer = base_model.get_layer(last_conv_layer_name)

    inputs = base_model.input
    conv_output = base_model.get_layer(last_conv_layer_name).output
    x = base_model.output
    top_layers = model.layers[1:] if isinstance(model, tf.keras.Sequential) else model.layers[2:]
    for layer in top_layers:
        x = layer(x)
    final_output = x
    grad_model = tf.keras.models.Model(inputs=[inputs], outputs=[conv_output, final_output])
    return grad_model

# Grad-CAM heatmap function with enhanced debugging
def get_gradcam_heatmap(grad_model, img_array):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        print("Warning: Gradients are None.")
        return None, predictions[0, 0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(conv_outputs, pooled_grads), axis=-1)
    heatmap = np.maximum(heatmap, 0)
    print(f"Heatmap stats - Min: {np.min(heatmap):.4f}, Max: {np.max(heatmap):.4f}, Mean: {np.mean(heatmap):.4f}")
    print(f"Gradient stats - Min: {np.min(grads):.4f}, Max: {np.max(grads):.4f}, Mean: {np.mean(grads):.4f}")
    if np.max(heatmap) < 1e-6:
        print("Warning: Heatmap max is too low, skipping normalization.")
        return None, predictions[0, 0]
    heatmap = heatmap / (np.max(heatmap) + 1e-10)
    return heatmap, predictions[0, 0]

# Load test generator
val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = val_datagen.flow_from_directory(
    '/content/chest_xray/chest_xray/test',
    target_size=(128, 128),
    batch_size=1,
    class_mode='binary',
    shuffle=False
)
print(f"Test generator loaded: {test_generator.samples} samples")
print(f"Class indices: {test_generator.class_indices}")

# Create Grad-CAM model
grad_model = create_gradcam_model(model)


normal_imgs, pneumonia_imgs = [], []
normal_labels, pneumonia_labels = [], []
normal_count, pneumonia_count = 0, 0
for i in range(test_generator.samples):
    img, label = next(test_generator)
    if label[0] == 0 and normal_count < 2:  # NORMAL
        normal_imgs.append(img)
        normal_labels.append(label[0])
        normal_count += 1
    elif label[0] == 1 and pneumonia_count < 2:  # PNEUMONIA
        pneumonia_imgs.append(img)
        pneumonia_labels.append(label[0])
        pneumonia_count += 1
    if normal_count == 2 and pneumonia_count == 2:
        break

if len(normal_imgs) < 2 or len(pneumonia_imgs) < 2:
    raise ValueError("Could not find enough NORMAL or PNEUMONIA images in test set.")

# Generate and plot heatmaps for two images per class
for idx in range(2):
    # Generate heatmaps
    print(f"\nProcessing NORMAL image {idx+1}")
    normal_heatmap, normal_pred = get_gradcam_heatmap(grad_model, normal_imgs[idx])
    if normal_heatmap is None:
        print(f"Skipping NORMAL image {idx+1} due to invalid heatmap.")
        continue
    print(f"Processing PNEUMONIA image {idx+1}")
    pneumonia_heatmap, pneumonia_pred = get_gradcam_heatmap(grad_model, pneumonia_imgs[idx])
    if pneumonia_heatmap is None:
        print(f"Skipping PNEUMONIA image {idx+1} due to invalid heatmap.")
        continue

    # Prepare images and heatmaps
    normal_img_display = normal_imgs[idx][0] * 255
    pneumonia_img_display = pneumonia_imgs[idx][0] * 255

    normal_heatmap = cv2.resize(normal_heatmap, (128, 128))
    normal_heatmap = np.uint8(255 * normal_heatmap)
    normal_heatmap = cv2.applyColorMap(normal_heatmap, cv2.COLORMAP_JET)
    normal_superimposed = normal_heatmap * 0.6 + normal_img_display * 0.4

    pneumonia_heatmap = cv2.resize(pneumonia_heatmap, (128, 128))
    pneumonia_heatmap = np.uint8(255 * pneumonia_heatmap)
    pneumonia_heatmap = cv2.applyColorMap(pneumonia_heatmap, cv2.COLORMAP_JET)
    pneumonia_superimposed = pneumonia_heatmap * 0.6 + pneumonia_img_display * 0.4

    # Plot side by side
    plt.figure(figsize=(15, 5))

    plt.subplot(2, 2, 1)
    plt.imshow(normal_img_display.astype('uint8'))
    plt.title(f"Normal Image {idx+1}\nPred: {'NORMAL' if normal_pred < 0.5 else 'PNEUMONIA'} (Conf: {normal_pred:.4f})")
    plt.axis('off')

    plt.subplot(2, 2, 2)
    plt.imshow(normal_superimposed.astype('uint8'))
    plt.title(f"Normal Grad-CAM Heatmap {idx+1}")
    plt.axis('off')

    plt.subplot(2, 2, 3)
    plt.imshow(pneumonia_img_display.astype('uint8'))
    plt.title(f"Pneumonia Image {idx+1}\nPred: {'PNEUMONIA' if pneumonia_pred > 0.5 else 'NORMAL'} (Conf: {pneumonia_pred:.4f})")
    plt.axis('off')

    plt.subplot(2, 2, 4)
    plt.imshow(pneumonia_superimposed.astype('uint8'))
    plt.title(f"Pneumonia Grad-CAM Heatmap {idx+1}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Clear Keras session
tf.keras.backend.clear_session()
print("Keras session cleared.")

# Load the trained model
try:
    model = tf.keras.models.load_model('enhanced_pneumonia_model.keras')
    print("Model loaded successfully.")
    dummy_input = np.zeros((1, 128, 128, 3), dtype=np.float32)
    model(dummy_input)
except Exception as e:
    print(f"Error loading model: {e}")
    print("Rebuilding model as fallback...")
    base_model = tf.keras.applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    inputs = tf.keras.Input(shape=(128, 128, 3))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Grad-CAM model with a shallower layer
def create_gradcam_model(model, last_conv_layer_name='block_7_project'):
    base_model = model.layers[0] if isinstance(model, tf.keras.Sequential) else model.layers[1]
    try:
        conv_layer = base_model.get_layer(last_conv_layer_name)
        print(f"Using layer: {last_conv_layer_name}")
    except ValueError:
        print(f"Layer {last_conv_layer_name} not found. Searching for last conv layer...")
        last_conv_layer_name = None
        for layer in base_model.layers[::-1]:
            if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
                last_conv_layer_name = layer.name
                print(f"Found conv layer: {last_conv_layer_name}")
                break
        if last_conv_layer_name is None:
            raise ValueError("No convolutional layer found in MobileNetV2.")
        conv_layer = base_model.get_layer(last_conv_layer_name)

    inputs = base_model.input
    conv_output = base_model.get_layer(last_conv_layer_name).output
    x = base_model.output
    top_layers = model.layers[1:] if isinstance(model, tf.keras.Sequential) else model.layers[2:]
    for layer in top_layers:
        x = layer(x)
    final_output = x
    grad_model = tf.keras.models.Model(inputs=[inputs], outputs=[conv_output, final_output])
    return grad_model

# Grad-CAM heatmap function with enhanced debugging
def get_gradcam_heatmap(grad_model, img_array):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        print("Warning: Gradients are None.")
        return None, predictions[0, 0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(conv_outputs, pooled_grads), axis=-1)
    heatmap = np.maximum(heatmap, 0)
    print(f"Heatmap stats - Min: {np.min(heatmap):.4f}, Max: {np.max(heatmap):.4f}, Mean: {np.mean(heatmap):.4f}")
    print(f"Gradient stats - Min: {np.min(grads):.4f}, Max: {np.max(grads):.4f}, Mean: {np.mean(grads):.4f}")
    print(f"Prediction: {predictions[0, 0]:.4f}")
    if np.max(heatmap) < 1e-5:
        print("Warning: Heatmap max is too low, skipping normalization.")
        return None, predictions[0, 0]
    heatmap = heatmap / (np.max(heatmap) + 1e-10)
    return heatmap, predictions[0, 0]

# Load test generator
val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = val_datagen.flow_from_directory(
    '/content/chest_xray/chest_xray/test',
    target_size=(128, 128),
    batch_size=1,
    class_mode='binary',
    shuffle=False
)
print(f"Test generator loaded: {test_generator.samples} samples")
print(f"Class indices: {test_generator.class_indices}")

# Create Grad-CAM model
grad_model = create_gradcam_model(model)

# Collect two NORMAL and two PNEUMONIA images
normal_imgs, pneumonia_imgs = [], []
normal_labels, pneumonia_labels = [], []
normal_count, pneumonia_count = 0, 0
for i in range(test_generator.samples):
    img, label = next(test_generator)
    if label[0] == 0 and normal_count < 2:
        normal_imgs.append(img)
        normal_labels.append(label[0])
        normal_count += 1
    elif label[0] == 1 and pneumonia_count < 2:
        pneumonia_imgs.append(img)
        pneumonia_labels.append(label[0])
        pneumonia_count += 1
    if normal_count == 2 and pneumonia_count == 2:
        break

if len(normal_imgs) < 2 or len(pneumonia_imgs) < 2:
    raise ValueError("Could not find enough NORMAL or PNEUMONIA images in test set.")

# Generate and plot heatmaps for two images per class
for idx in range(2):
    # Generate heatmaps
    print(f"\nProcessing NORMAL image {idx+1}")
    normal_heatmap, normal_pred = get_gradcam_heatmap(grad_model, normal_imgs[idx])
    if normal_heatmap is None:
        print(f"Skipping NORMAL image {idx+1} due to invalid heatmap.")
        continue
    print(f"Processing PNEUMONIA image {idx+1}")
    pneumonia_heatmap, pneumonia_pred = get_gradcam_heatmap(grad_model, pneumonia_imgs[idx])
    if pneumonia_heatmap is None:
        print(f"Skipping PNEUMONIA image {idx+1} due to invalid heatmap.")
        continue

    # Prepare images and heatmaps
    normal_img_display = normal_imgs[idx][0] * 255
    pneumonia_img_display = pneumonia_imgs[idx][0] * 255

    normal_heatmap = cv2.resize(normal_heatmap, (128, 128))
    normal_heatmap = np.uint8(255 * normal_heatmap)
    normal_heatmap = cv2.applyColorMap(normal_heatmap, cv2.COLORMAP_JET)
    normal_superimposed = normal_heatmap * 0.6 + normal_img_display * 0.4

    pneumonia_heatmap = cv2.resize(pneumonia_heatmap, (128, 128))
    pneumonia_heatmap = np.uint8(255 * pneumonia_heatmap)
    pneumonia_heatmap = cv2.applyColorMap(pneumonia_heatmap, cv2.COLORMAP_JET)
    pneumonia_superimposed = pneumonia_heatmap * 0.6 + pneumonia_img_display * 0.4

    # Plot side by side
    plt.figure(figsize=(15, 5))

    plt.subplot(2, 2, 1)
    plt.imshow(normal_img_display.astype('uint8'))
    plt.title(f"Normal Image {idx+1}\nPred: {'NORMAL' if normal_pred < 0.5 else 'PNEUMONIA'} (Conf: {normal_pred:.4f})")
    plt.axis('off')

    plt.subplot(2, 2, 2)
    plt.imshow(normal_superimposed.astype('uint8'))
    plt.title(f"Normal Grad-CAM Heatmap {idx+1}")
    plt.axis('off')

    plt.subplot(2, 2, 3)
    plt.imshow(pneumonia_img_display.astype('uint8'))
    plt.title(f"Pneumonia Image {idx+1}\nPred: {'PNEUMONIA' if pneumonia_pred > 0.5 else 'NORMAL'} (Conf: {pneumonia_pred:.4f})")
    plt.axis('off')

    plt.subplot(2, 2, 4)
    plt.imshow(pneumonia_superimposed.astype('uint8'))
    plt.title(f"Pneumonia Grad-CAM Heatmap {idx+1}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

#More training

In [ ]:
import os
from tensorflow.keras.callbacks import ReduceLROnPlateau

base_dir = '/content/chest_xray/chest_xray'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Verify directories
print("Train dir exists:", os.path.exists(train_dir))
print("Test dir exists:", os.path.exists(test_dir))
if os.path.exists(train_dir):
    print("Train subdirs:", os.listdir(train_dir))

sub_train_dir = '/content/sub_train'
sub_val_dir = '/content/sub_val'
os.makedirs(sub_train_dir, exist_ok=True)
os.makedirs(sub_val_dir, exist_ok=True)

def subsample_dir(src_dir, dest_dir, class_name, num_samples):
    src_path = os.path.join(src_dir, class_name)
    dest_path = os.path.join(dest_dir, class_name)
    if not os.path.exists(src_path):
        print(f"Error: {src_path} not found. Skipping {class_name}.")
        return False
    os.makedirs(dest_path, exist_ok=True)
    files = os.listdir(src_path)
    if not files:
        print(f"Error: No files in {src_path}.")
        return False
    random.shuffle(files)
    for file in files[:num_samples]:
        shutil.copy(os.path.join(src_path, file), os.path.join(dest_path, file))
    print(f"Copied {min(num_samples, len(files))} files for {class_name} to {dest_path}")
    return True

classes = ['NORMAL', 'PNEUMONIA']
success = True
for cls in classes:
    if not subsample_dir(train_dir, sub_train_dir, cls, 250):
        success = False
    if not subsample_dir(train_dir, sub_val_dir, cls, 50):
        success = False
if not success:
    raise FileNotFoundError("Failed to subsample dataset. Check directory structure.")

img_size = (128, 128)
batch_size = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    sub_train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary'
)
val_generator = val_datagen.flow_from_directory(
    sub_val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary'
)
test_generator = val_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False
)
print(f"Train samples: {train_generator.samples}, Val samples: {val_generator.samples}, Test samples: {test_generator.samples}")
if train_generator.samples == 0 or val_generator.samples == 0:
    raise ValueError("No images found in train/val directories. Check subsampling.")

def plot_class_distribution(generator, title):
    labels = generator.classes
    unique, counts = np.unique(labels, return_counts=True)
    df = pd.DataFrame({'Class': ['NORMAL', 'PNEUMONIA'], 'Count': counts})
    plt.figure(figsize=(6, 4))
    sns.barplot(x='Class', y='Count', hue='Class', data=df, palette=['#1f77b4', '#ff7f0e'], legend=False)
    plt.title(title)
    plt.show()

plot_class_distribution(train_generator, 'Train Class Distribution')
plot_class_distribution(val_generator, 'Val Class Distribution')


def build_model(hp):
    base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base_model.trainable = True
    for layer in base_model.layers[:100]:  # Freeze first 100 layers
        layer.trainable = False

    model = Sequential([
        base_model,
        Flatten(),
        Dense(hp.Int('units', 32, 128, step=32), activation='relu'),
        Dropout(hp.Float('dropout', 0.3, 0.7, step=0.1)),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(hp.Float('lr', 1e-5, 1e-3, sampling='log')),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    return model


!pip install keras-tuner

# Tune hyperparameters
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=5,
    directory='tuner_dir',
    project_name='pneumonia_enhanced'
)
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)
]
tuner.search(train_generator, epochs=5, validation_data=val_generator, callbacks=callbacks)


best_model = tuner.get_best_models(num_models=1)[0]

class_weight = {0: 1.5, 1: 1.0}
print("Starting fine-tuning with class weights...")
history = best_model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weight
)
print("Training complete.")


plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc', color='#1f77b4')
plt.plot(history.history['val_accuracy'], label='Val Acc', color='#ff7f0e')
plt.title('Model Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='#1f77b4')
plt.plot(history.history['val_loss'], label='Val Loss', color='#ff7f0e')
plt.title('Model Loss')
plt.legend()
plt.show()


test_loss, test_acc, test_precision, test_recall = best_model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc:.2f}, Precision: {test_precision:.2f}, Recall: {test_recall:.2f}")

# Confusion Matrix
y_pred = (best_model.predict(test_generator) > 0.5).astype(int)
y_true = test_generator.classes
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['NORMAL', 'PNEUMONIA'], yticklabels=['NORMAL', 'PNEUMONIA'])
plt.title('Confusion Matrix')
plt.show()
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

best_model.save('enhanced_pneumonia_model.keras')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from flask import Flask, request, jsonify
import io
from PIL import Image

# Fix Seaborn plot function
def plot_class_distribution(generator, title):
    labels = generator.classes
    unique, counts = np.unique(labels, return_counts=True)
    df = pd.DataFrame({'Class': ['NORMAL', 'PNEUMONIA'], 'Count': counts})
    plt.figure(figsize=(6, 4))
    sns.barplot(x='Class', y='Count', hue='Class', data=df, palette=['#1f77b4', '#ff7f0e'], legend=False)
    plt.title(title)
    plt.show()

# Plot class distributions
plot_class_distribution(train_generator, 'Train Class Distribution')
plot_class_distribution(val_generator, 'Val Class Distribution')

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True)
]

# Train model (3 epochs, ~8-10 mins on GPU)
print("Starting training...")

class_weight = {0: 1.5, 1: 1.0}  # Higher weight for NORMAL to balance recall

# Update model.fit
history = model.fit(
    train_generator,
    epochs=3,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weight  # Add weights
)
print("Training complete.")

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc', color='#1f77b4')
plt.plot(history.history['val_accuracy'], label='Val Acc', color='#ff7f0e')
plt.title('Model Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='#1f77b4')
plt.plot(history.history['val_loss'], label='Val Loss', color='#ff7f0e')
plt.title('Model Loss')
plt.legend()
plt.show()

# Evaluate on test set
model.load_weights('best_model.h5')
test_loss, test_acc, test_precision, test_recall = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc:.2f}, Precision: {test_precision:.2f}, Recall: {test_recall:.2f}")

# Confusion matrix
y_pred = (model.predict(test_generator) > 0.5).astype(int)
y_true = test_generator.classes
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['NORMAL', 'PNEUMONIA'], yticklabels=['NORMAL', 'PNEUMONIA'])
plt.title('Confusion Matrix')
plt.show()
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))


model.save('pneumonia_model.h5')


app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    file = request.files['image']
    img = Image.open(io.BytesIO(file.read())).resize((128, 128))
    img = np.array(img) / 255.0
    if img.shape[-1] == 4:
        img = img[:, :, :3]
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img)[0][0]
    return jsonify({'prediction': 'PNEUMONIA' if pred > 0.5 else 'NORMAL', 'confidence': float(pred)})

